<a href="https://colab.research.google.com/github/Israzuba0023/Err00723/blob/main/Titanic%20-%20Machine%20Learning%20from%20Disaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sessão de Desafio Kaggle
Título do desafio:
#Titanic - Aprendizado de máquina apartir de  desastres

- Link do Kaggle: h7ps://www.kaggle.com/compe44ons/4tanic
Obje:vo: Use suas habilidades de ciência de dados e aprendizado de máquina para construir um modelo que preveja quais passageiros sobreviveram ao naufrágio do Titanic.


In [17]:
# Importações básicas
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#  Carregamento e Inspeção dos Dados

In [18]:
# Carregar os dados
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Guardar o PassengerId do conjunto de teste para a submissão final
passenger_ids = df_test['PassengerId']

# Inspecionar as primeiras linhas e valores faltantes no treino
print("Dados de Treino:")
df_train.info()

print("\nDados de Teste:")
df_test.info()

Dados de Treino:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Dados de Teste:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418

#Pré-processamento Simples (Para uma Rápida Submissão)

#Unir DataFrames (Opcional, mas Útil)

- Para garantir que o mesmo tratamento seja feito em treino e teste, vamos concatená-los (removendo a coluna Survived do treino primeiro):


In [19]:
# Target (Y) e Features (X)
y_train = df_train['Survived']
X_train = df_train.drop('Survived', axis=1)


#Tratamento de Valores Faltantes (Imputação)

    Age (Idade): Preencher com a mediana.

    Fare (Tarifa): Preencher o valor faltante (apenas no df_test) com a mediana.

    Embarked (Embarque): Preencher os valores faltantes (apenas no df_train) com o modo (valor mais frequente).

In [20]:
# Imputação no TREINO
df_train['Age'].fillna(df_train['Age'].median(), inplace=True)
df_train['Embarked'].fillna(df_train['Embarked'].mode()[0], inplace=True)

# Imputação no TESTE
df_test['Age'].fillna(df_test['Age'].median(), inplace=True)
df_test['Fare'].fillna(df_test['Fare'].median(), inplace=True)

/tmp/ipython-input-49489618.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_train['Age'].fillna(df_train['Age'].median(), inplace=True)
/tmp/ipython-input-49489618.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True

# Conversão de Variáveis Categóricas (One-Hot Encoding)

- O modelo de Machine Learning (Regressão Logística) precisa de números, então convertemos Sex (Sexo) e Embarked (Embarque).

In [21]:
# Focaremos nas features Pclass, Sex, Age, SibSp, Parch, Fare, Embarked
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

# Usamos get_dummies para Sex e Embarked
X_train_processed = pd.get_dummies(df_train[features], columns=['Sex', 'Embarked'], drop_first=True)
X_test_processed = pd.get_dummies(df_test[features], columns=['Sex', 'Embarked'], drop_first=True)

# Garantir que os dataframes de treino e teste tenham as mesmas colunas
missing_cols = set(X_train_processed.columns) - set(X_test_processed.columns)
for c in missing_cols:
    X_test_processed[c] = 0
X_test_processed = X_test_processed[X_train_processed.columns]

print("\nColunas de Treino Prontas:", X_train_processed.columns.tolist())


Colunas de Treino Prontas: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']


#Treinamento do Modelo

In [22]:
# Inicializar o modelo
model = LogisticRegression(solver='liblinear', random_state=42)

# Treinar o modelo
model.fit(X_train_processed, y_train)

# Fazer a previsão no conjunto de teste (X_test_processed)
predictions = model.predict(X_test_processed)

# verificar a performance no conjunto de treino
train_pred = model.predict(X_train_processed)
train_accuracy = accuracy_score(y_train, train_pred)
print(f"\nAcurácia no Treino: {train_accuracy:.4f}")


Acurácia no Treino: 0.8013


# Criação e Envio da Submissão

- O Kaggle exige um arquivo CSV com as colunas PassengerId e Survived.

In [24]:
# Criar o DataFrame de submissão
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': predictions
})

# O Kaggle precisa que a coluna 'Survived' seja de inteiros (0 ou 1)
submission['Survived'] = submission['Survived'].astype(int)

# Salvar o arquivo CSV
submission.to_csv('submission_primeira_tentativa.csv', index=False)

print("\nArquivo de submissão criado: 'submission_primeira_tentativa.csv'")
print("\n--- Exemplo do arquivo de submissão ---")
print(submission.head())


Arquivo de submissão criado: 'submission_primeira_tentativa.csv'

--- Exemplo do arquivo de submissão ---
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1


#Modelo1: resultado

- Sua Primeira Entrada!Bem-vindo à tabela de classificação! Sua pontuação representa a precisão do seu envio. Por exemplo, uma pontuação de 0,7 nesta competição indica que você previu a sobrevivência do Titanic corretamente para 70% das pessoas.

# Modelo2

# Preparação do Ambiente e Consolidação de Dados

Vamos começar de onde paramos, mas agora unindo os DataFrames.

In [25]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier # Novo modelo
from sklearn.model_selection import cross_val_score # Para validação

# 1. Carregar os Dados
# Certifique-se de que os arquivos train.csv e test.csv estão no diretório /content/
try:
    df_train = pd.read_csv('/content/train.csv')
    df_test = pd.read_csv('/content/test.csv')
except FileNotFoundError:
    print("Certifique-se de que os arquivos train.csv e test.csv estão no diretório /content/ ou no caminho correto do seu Google Drive.")
    # Se estiver usando o Google Drive, descomente e ajuste o caminho abaixo:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # drive_path = '/content/drive/MyDrive/Seu/Caminho/Para/Os/Arquivos' # Ajuste este caminho
    # df_train = pd.read_csv(f'{drive_path}/train.csv')
    # df_test = pd.read_csv(f'{drive_path}/test.csv')


# Salvar o PassengerId do teste e o Target do treino
passenger_ids = df_test['PassengerId']
Y_train = df_train['Survived']

# Remover a coluna 'Survived' do treino para concatenar
X_train = df_train.drop('Survived', axis=1)

# Concatenar para tratar ambas as bases de uma vez
# O 'df_test' não tem a coluna 'Survived', e o 'X_train' também não.
# Garantimos que a coluna 'Ticket' esteja presente para a feature engineering futura
df_all = pd.concat([X_train, df_test], ignore_index=True)

# Vamos verificar a primeira linha para confirmar a união (apenas para debug)
print("DataFrame combinado (df_all) - Primeiras linhas:")
print(df_all.head())

DataFrame combinado (df_all) - Primeiras linhas:
   PassengerId  Pclass                                               Name  \
0            1       3                            Braund, Mr. Owen Harris   
1            2       1  Cumings, Mrs. John Bradley (Florence Briggs Th...   
2            3       3                             Heikkinen, Miss. Laina   
3            4       1       Futrelle, Mrs. Jacques Heath (Lily May Peel)   
4            5       3                           Allen, Mr. William Henry   

      Sex   Age  SibSp  Parch            Ticket     Fare Cabin Embarked  
0    male  22.0      1      0         A/5 21171   7.2500   NaN        S  
1  female  38.0      1      0          PC 17599  71.2833   C85        C  
2  female  26.0      0      0  STON/O2. 3101282   7.9250   NaN        S  
3  female  35.0      1      0            113803  53.1000  C123        S  
4    male  35.0      0      0            373450   8.0500   NaN        S  


# Engenharia de Recursos Avançada

Agora aplicar as melhorias de feature engineering no df_all:

- 2.1. Feature: Título

Extrair o título (Mr., Mrs., Miss, Master, etc.) do nome, pois eles estão altamente correlacionados com o sexo, status social e sobrevivência.

In [26]:
# Extrair o Título
df_all['Title'] = df_all['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Simplificar títulos raros em uma categoria 'Rare'
rare_titles = ['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df_all['Title'] = df_all['Title'].replace(rare_titles, 'Rare')
df_all['Title'] = df_all['Title'].replace('Mlle', 'Miss')
df_all['Title'] = df_all['Title'].replace('Ms', 'Miss')
df_all['Title'] = df_all['Title'].replace('Mme', 'Mrs')

# A coluna 'Title' é agora uma nova feature categórica
print("\nContagem dos Títulos após simplificação:")
print(df_all['Title'].value_counts())


Contagem dos Títulos após simplificação:
Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64


<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-1898486860.py:2: SyntaxWarning: invalid escape sequence '\.'
  df_all['Title'] = df_all['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


- Feature: Tamanho da Família (FamilySize)

Pessoas que viajavam sozinhas (ou em famílias muito grandes/pequenas) podem ter tido diferentes taxas de sobrevivência.

In [27]:
# SibSp (irmãos/cônjuges) + Parch (pais/filhos) + 1 (o próprio passageiro)
df_all['FamilySize'] = df_all['SibSp'] + df_all['Parch'] + 1

# Criar uma feature categórica para o tamanho da família
df_all['IsAlone'] = 0
df_all.loc[df_all['FamilySize'] == 1, 'IsAlone'] = 1 # 1 se estiver sozinho

- Feature: Nível da Cabine (Cabin Deck)

A primeira letra da cabine indica o convés, o que sugere a localização do passageiro no navio (e, portanto, a proximidade aos botes salva-vidas).

In [28]:
# A primeira letra da Cabine é o Deck. Preencher NaNs com 'Unknown'
df_all['Deck'] = df_all['Cabin'].str[0].fillna('Unknown')
df_all = df_all.drop('Cabin', axis=1) # Cabin original não é mais necessária

#Tratamento de Valores Faltantes (Imputação e Preenchimento Final)

Faremos a imputação final usando os novos insights:

In [29]:
# 1. Imputar AGE: Usamos a mediana da IDADE POR TÍTULO
# Isso é mais preciso do que apenas a mediana geral
df_all['Age'] = df_all.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

# 2. Imputar FARE (faltando 1 no teste): Preencher com a mediana geral
df_all['Fare'].fillna(df_all['Fare'].median(), inplace=True)

# 3. Imputar EMBARKED (faltando 2 no treino): Preencher com o modo
df_all['Embarked'].fillna(df_all['Embarked'].mode()[0], inplace=True)

# Remover colunas que não usaremos mais
df_all = df_all.drop(['Name', 'Ticket', 'SibSp', 'Parch'], axis=1)

print("\nValores faltantes restantes (deve ser 0 em tudo):")
print(df_all.isnull().sum())


Valores faltantes restantes (deve ser 0 em tudo):
PassengerId    0
Pclass         0
Sex            0
Age            0
Fare           0
Embarked       0
Title          0
FamilySize     0
IsAlone        0
Deck           0
dtype: int64


/tmp/ipython-input-1624422757.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_all['Fare'].fillna(df_all['Fare'].median(), inplace=True)
/tmp/ipython-input-1624422757.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tr

# Codificação e Separação de Dados

 - One-Hot Encoding Final

Transformar todas as variáveis categóricas restantes (Sex, Embarked, Title, Deck) em numéricas.

In [30]:
# Aplicar One-Hot Encoding em todas as variáveis categóricas
X_processed = pd.get_dummies(df_all, columns=['Pclass', 'Sex', 'Embarked', 'Title', 'Deck'])

# Remover colunas originais que viraram dummies e o PassengerId (não é um preditor)
X_processed = X_processed.drop('PassengerId', axis=1)

- Separar Treino e Teste

Usamos o índice que salvamos antes para separa

In [31]:
train_len = len(df_train) # Sabemos que o treino tem 891 linhas

X_train_final = X_processed.iloc[:train_len]
X_test_final = X_processed.iloc[train_len:]

# Verificar as formas (shape) para confirmar
print(f"\nFormato de X_train_final: {X_train_final.shape}")
print(f"Formato de X_test_final: {X_test_final.shape}")


Formato de X_train_final: (891, 26)
Formato de X_test_final: (418, 26)


# Treinamento com Random Forest

O Random Forest é um modelo de ensemble muito mais poderoso que a Regressão Logística, geralmente dando um grande salto de precisão.

In [32]:
# Inicializar o modelo Random Forest
# Ajustamos alguns hiperparâmetros básicos para melhor performance
model_rf = RandomForestClassifier(n_estimators=100, max_depth=7, min_samples_split=8, random_state=42)

# Treinar o modelo
model_rf.fit(X_train_final, Y_train)

# Fazer a previsão no conjunto de teste
predictions_rf = model_rf.predict(X_test_final)

# (Opcional) Usar Cross-Validation para estimar a precisão real no treino
cv_scores = cross_val_score(model_rf, X_train_final, Y_train, cv=5)
print(f"\nPontuação Média de Cross-Validation (5-folds): {cv_scores.mean():.4f}")
# Este número é uma excelente estimativa do que você verá no Kaggle!


Pontuação Média de Cross-Validation (5-folds): 0.8339


# Submissão Final

Criar o arquivo CSV para o Kaggle:

In [36]:
# Criar o DataFrame de submissão
submission_rf = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': predictions_rf
})

# Garantir que 'Survived' seja de inteiros (0 ou 1)
submission_rf['Survived'] = submission_rf['Survived'].astype(int)

# Salvar o arquivo CSV
submission_rf.to_csv('submission_random_forest_final.csv', index=False)

print("\nSUCESSO! Arquivo de submissão Random Forest criado.")
print("Baixe 'submission_random_forest_final.csv' e submeta no Kaggle.")


SUCESSO! Arquivo de submissão Random Forest criado.
Baixe 'submission_random_forest_final.csv' e submeta no Kaggle.


*Resultado do modelo 2:*  Sua Melhor Entrada!Sua apresentação mais recente obteve 0.77511, o que é uma melhoria em relação à sua pontuação anterior de 0,76076. Ótimo trabalho!